<h1 style="text-align: center;">Telkomsel Exploratory Data Analysis</h1>
<h3 style="text-align: center;">Muhammad Rafi Andrianto</h3>
<h3 style="text-align: center;">Yoanita Dwi Harlandi</h3>

---

## **Section 1. Business Context**

**1.1 Context**

**1.2 Problem Statements**

**1.3 Key Objective**

## **Section 2. Data Understanding**

**2.1 General Information**

**2.2 Feature Information**

**2.3 Statistics Summary**

## **Section 3. Data Cleaning**

In [2]:
import pandas as pd
import numpy as np

In [3]:
sub_raw = pd.read_csv('../data/raw/tsel_subscribers.csv')
pkg_raw = pd.read_csv('../data/raw/tsel_packages.csv')
usage_raw = pd.read_csv('../data/raw/tsel_data_usage.csv')

print(f"Subscribers Shape : {sub_raw.shape}")
print(f"Packages Shape    : {pkg_raw.shape}")
print(f"Data Usage Shape  : {usage_raw.shape}")

Subscribers Shape : (35000, 4)
Packages Shape    : (10, 3)
Data Usage Shape  : (300000, 7)


**3.1 Missing Values**

In [4]:
print("--- tsel_subscribers ---")
print(sub_raw.isna().sum())

print("\n--- tsel_data_usage ---")
print(usage_raw.isna().sum())

invalid_nik = sub_raw[sub_raw['nik_kk_status'].isna()]
pct_invalid_nik = (len(invalid_nik) / len(sub_raw)) * 100

print(f"\nPersentase NIK/KK Status bermasalah/missing: {pct_invalid_nik:.2f}%")

--- tsel_subscribers ---
msisdn                 0
activation_date        0
nik_kk_status       5202
registered_imei    24470
dtype: int64

--- tsel_data_usage ---
session_id      0
msisdn          0
package_code    0
used_imei       0
session_time    0
network_type    0
payload_mb      0
dtype: int64

Persentase NIK/KK Status bermasalah/missing: 14.86%


**3.2 Duplicated Values**

In [5]:
print(f"Duplikat di tsel_subscribers : {sub_raw.duplicated().sum()}")
print(f"Duplikat di tsel_packages    : {pkg_raw.duplicated().sum()}")
print(f"Duplikat di tsel_data_usage  : {usage_raw.duplicated().sum()}")

Duplikat di tsel_subscribers : 0
Duplikat di tsel_packages    : 0
Duplikat di tsel_data_usage  : 0


**3.3 Identify Spelling Errors**

In [9]:
print(usage_raw['network_type'].value_counts())

network_type
4G        89981
LTE       44884
4G LTE    30339
5G        30075
4g        29833
WCDMA     15225
3G        15076
NR        14941
HSDPA     14905
5g        14741
Name: count, dtype: int64


**3.4 Identify Anomaly Values**

In [18]:
print(f"--- subscribers data type ---\n{sub_raw.dtypes}\n")
print(f"--- package data type ---\n{pkg_raw.dtypes}\n")
print(f"--- usage data type ---\n{usage_raw.dtypes}")

--- subscribers data type ---
msisdn               int64
activation_date        str
nik_kk_status          str
registered_imei    float64
dtype: object

--- package data type ---
package_code      str
package_name      str
price           int64
dtype: object

--- usage data type ---
session_id          str
msisdn            int64
package_code        str
used_imei         int64
session_time        str
network_type        str
payload_mb      float64
dtype: object


In [ ]:
sub_dt = sub_raw[['msisdn', 'activation_date']].copy()
sub_dt['activation_date'] = pd.to_datetime(sub_dt['activation_date'])

usage_dt = usage_raw[['msisdn', 'session_time', 'payload_mb', 'session_id']].copy()
usage_dt['session_time'] = pd.to_datetime(usage_dt['session_time'])

#merge untuk validasi tanggal
df_merge_time = pd.merge(usage_dt, sub_dt, on='msisdn', how='left')
anomaly_time = df_merge_time[df_merge_time['session_time'] < df_merge_time['activation_date']]

print(f"jumlah sesi internet SEBELUM tanggal aktivasi: {len(anomaly_time)}")
anomaly_time.head()

jumlah sesi internet SEBELUM tanggal aktivasi: 12008


,msisdn,session_time,payload_mb,session_id,activation_date
26,82254454198,2023-04-18,133.47,CDR000000027,2023-04-20
34,82222192600,2023-10-04,758.03,CDR000000035,2023-10-11
63,85272766900,2023-04-07,1090.95,CDR000000064,2023-04-09
104,82120468327,2023-10-17,760.71,CDR000000105,2023-10-18
124,81188580169,2023-02-15,409.70,CDR000000125,2023-02-18


In [48]:
print(usage_raw['payload_mb'].describe())

count    300000.000000
mean       6159.144994
std       71485.176443
min        -150.500000
25%         506.337500
50%        1024.020000
75%        1541.155000
max      999999.900000
Name: payload_mb, dtype: float64


In [58]:
outlier_extreme = usage_raw[usage_raw['payload_mb'] >= 50000] # max sementara 50 GB per sesi
print(f"\njumlah sesi yang melebihi 50,000 MB: {len(outlier_extreme)}")
print("top nilai terbesar payload_mb:")
print(usage_raw['payload_mb'].nlargest())


jumlah sesi yang melebihi 50,000 MB: 1544
top nilai terbesar payload_mb:
2       999999.9
451     999999.9
1118    999999.9
1353    999999.9
1529    999999.9
Name: payload_mb, dtype: float64


**3.5 Creating Clean Datasets**

In [62]:
sub_clean = sub_raw.copy()
pkg_clean = pkg_raw.copy()
usage_clean = usage_raw.copy()

sub_clean['activation_date'] = pd.to_datetime(sub_clean['activation_date'])
usage_clean['session_time'] = pd.to_datetime(usage_clean['session_time'])

In [72]:
#Tangani Missing Value pada NIK/KK Status (Ubah NaN jadi missing/unregis)
sub_clean['nik_kk_status'] = sub_clean['nik_kk_status'].fillna('Missing/Unregistered')

In [75]:
#Standardisasi Network Type (mapping 10 variasi menjadi 3)
network_map = {
    '4G': '4G', 
    'LTE': '4G', 
    '4G LTE': '4G', 
    '4g': '4G',
    '5G': '5G', 
    'NR': '5G', 
    '5g': '5G',
    'WCDMA': '3G', 
    '3G': '3G', 
    'HSDPA': '3G'
}
usage_clean['network_type'] = usage_clean['network_type'].map(network_map)
print(usage_clean['network_type'].value_counts())

network_type
4G    195037
5G     59757
3G     45206
Name: count, dtype: int64


In [76]:
#Hapus outlier dan value aneh pada Payload MB (hapus nilai < 0 dan nilai extreme 999999.9)
usage_clean = usage_clean[(usage_clean['payload_mb'] >= 0) & (usage_clean['payload_mb'] < 999999.9)]

#Hapus sesi internet yang dipakai sebelum tanggal aktivasi kartu
#merge sementara dengan data activation_date untuk memfilter baris yang valid
df_merge_filter = pd.merge(usage_clean, sub_clean[['msisdn', 'activation_date']], on='msisdn', how='left')
usage_clean = df_merge_filter[df_merge_filter['session_time'] >= df_merge_filter['activation_date']].drop(columns=['activation_date'])

In [84]:
print(usage_clean['payload_mb'].describe())

count    285123.000000
mean       1023.542969
std         590.941434
min           0.010000
25%         511.490000
50%        1023.800000
75%        1535.645000
max        2047.980000
Name: payload_mb, dtype: float64


In [87]:
#save to csv
sub_clean.to_csv('../data/cleaned/clean_subscribers.csv', index=False)
usage_clean.to_csv('../data/cleaned/clean_data_usage.csv', index=False)
pkg_raw.to_csv('../data/cleaned/clean_package.csv', index=False)

## **Section 4. Analytics**

### **4.1 Question 1**

### **4.2 Question 2**

## **Section 5. Conclusion and Recommendation**

**5.1 Conclusion**

**5.2 Recommendation**